In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
Enhanced Title-Retrieval Pipeline v2
=====================================
Targets the specific weakness: Catchy (~0.4956 MRR@10) and Accessible (~0.3883 MRR@10)
underperform because:
  1. MNRL trained with in-batch negatives only -> never sees "hard" confusable negatives
  2. BM25 has low lexical overlap on Catchy titles (marketing language, wordplay)
  3. Reranker is zero-shot, never adapted to this domain's rewrite style
  4. No BGE retrieval instruction prefix on queries
  5. Naive score fusion (raw cosine ∪ raw BM25) instead of calibrated fusion

Fixes applied, in order of expected impact:
  A. Hard-negative mining via BM25 + MultipleNegativesRankingLoss with explicit negatives
  B. BGE query instruction prefix ("Represent this sentence for searching relevant passages:")
  C. Category-aware oversampling of Catchy/Accessible pairs during fine-tuning
  D. Fine-tune the cross-encoder reranker on (query, pos, hard-neg) triples
  E. Reciprocal Rank Fusion (RRF) instead of raw-score set union for candidate merging
  F. Wider candidate pool (top-50 instead of top-30) before reranking
  G. Per-category MRR reporting so you can verify the gap closes
"""

import os

# Must be set BEFORE torch is imported. On multi-GPU environments (e.g. Kaggle T4x2),
# PyTorch/sentence-transformers/bert_score can silently wrap models in
# nn.DataParallel. That wrapper only forwards forward() -- custom methods like
# CrossEncoder's .preprocess() or bert_score's internal calls aren't exposed on it,
# which is exactly what causes:
#   AttributeError: 'DataParallel' object has no attribute 'preprocess'
# Restricting to a single visible GPU avoids the auto-wrapping entirely.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import re
import random
import numpy as np
import pandas as pd
import torch
import faiss
from tqdm import tqdm
from torch.utils.data import DataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, InputExample, losses, CrossEncoder
import evaluate

# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing Enhanced Retrieval & Evaluation Pipeline v2 on: {DEVICE}")

# BGE asymmetric-retrieval instruction prefix (query side only, per BAAI docs)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

# Categories we know are weak — used for oversampling weight in training
HARD_CATEGORIES = {"Catchy", "Accessible"}
HARD_CATEGORY_OVERSAMPLE_FACTOR = 2  # duplicate hard-category pairs this many extra times

# ==========================================
# 1. DATA PREPARATION & CLEANING
# ==========================================
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


print("\n[1/7] Loading Datasets...")
train_df = pd.read_csv("/kaggle/input/datasets/akshithanaiduzoo/fifi2026/train.tsv", sep="\t")
val_df = pd.read_csv("/kaggle/input/datasets/akshithanaiduzoo/fifi2026/val.tsv", sep="\t")

for df in (train_df, val_df):
    df["original_title"] = df["original_title"].apply(clean_text)
    df["generated_title"] = df["generated_title"].apply(clean_text)

corpus = sorted(list(set(train_df["original_title"].dropna()).union(set(val_df["original_title"].dropna()))))
title2id = {t: i for i, t in enumerate(corpus)}
print(f"Total Corpus Size: {len(corpus):,} unique original titles")

if "category" in train_df.columns:
    print("Train category counts:\n", train_df["category"].value_counts())

# ==========================================
# 2. HARD NEGATIVE MINING (BM25-based)
# ==========================================
# Why: in-batch negatives are almost always trivially different from the positive.
# Mining BM25 top-k negatives (excluding the true positive) gives the encoder
# confusable-but-wrong examples, which is exactly what separates Catchy/Accessible
# titles (high semantic drift, so easy negatives don't teach the fine boundary).
print("\n[2/7] Mining Hard Negatives with BM25...")

bm25_corpus_tokens = [doc.lower().split() for doc in corpus]
bm25_global = BM25Okapi(bm25_corpus_tokens)

N_HARD_NEGATIVES = 4

def mine_hard_negatives(query_text, true_title, n=N_HARD_NEGATIVES):
    scores = bm25_global.get_scores(query_text.lower().split())
    top_idx = np.argsort(scores)[::-1][: n + 5]  # buffer in case positive is in there
    negs = []
    for idx in top_idx:
        cand = corpus[idx]
        if cand != true_title:
            negs.append(cand)
        if len(negs) >= n:
            break
    return negs


HARD_NEG_CACHE_PATH = "train_hard_negatives.pkl"
if not os.path.exists(HARD_NEG_CACHE_PATH):
    hard_negs_list = []
    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
        hard_negs_list.append(mine_hard_negatives(row["generated_title"], row["original_title"]))
    train_df["hard_negatives"] = hard_negs_list
    train_df.to_pickle(HARD_NEG_CACHE_PATH)
    print(f"Hard negatives mined and cached to '{HARD_NEG_CACHE_PATH}'")
else:
    train_df = pd.read_pickle(HARD_NEG_CACHE_PATH)
    print(f"Loaded cached hard negatives from '{HARD_NEG_CACHE_PATH}'")

# ==========================================
# 3. FINE-TUNE BI-ENCODER WITH HARD NEGATIVES + CATEGORY OVERSAMPLING
# ==========================================
MODEL_SAVE_PATH = "fine_tuned_bge_retriever_v2"

if not os.path.exists(MODEL_SAVE_PATH):
    print("\n[3/7] FINE-TUNING BI-ENCODER WITH HARD NEGATIVES...")

    train_examples = []
    for _, row in train_df.iterrows():
        query = QUERY_PREFIX + row["generated_title"]
        pos = row["original_title"]
        negs = row.get("hard_negatives", []) or []
        # InputExample texts=[anchor, positive, neg1, neg2, ...] is supported by MNRL:
        # negatives beyond index 1 are treated as additional (hard) negatives.
        texts = [query, pos] + list(negs)
        example = InputExample(texts=texts)

        weight = HARD_CATEGORY_OVERSAMPLE_FACTOR if row.get("category") in HARD_CATEGORIES else 1
        train_examples.extend([example] * weight)

    print(f"Effective training examples after oversampling: {len(train_examples):,}")

    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

    base_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
    # CachedMultipleNegativesRankingLoss increases the *effective* batch size (more
    # in-batch negatives) without blowing up GPU memory — improves contrastive signal.
    try:
        train_loss = losses.CachedMultipleNegativesRankingLoss(base_model, mini_batch_size=16)
    except AttributeError:
        train_loss = losses.MultipleNegativesRankingLoss(base_model)

    print("Starting 2 Epochs of MNRL Fine-Tuning with Hard Negatives...")
    base_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=2,
        warmup_steps=int(0.1 * len(train_dataloader) * 2),
        optimizer_params={"lr": 2e-5},
        output_path=MODEL_SAVE_PATH,
        show_progress_bar=True,
    )
    print(f"Fine-tuning complete! Model saved to '{MODEL_SAVE_PATH}'")
else:
    print(f"\n[3/7] Found existing fine-tuned model at '{MODEL_SAVE_PATH}'. Loading...")

dense_encoder = SentenceTransformer(MODEL_SAVE_PATH, device=DEVICE)

# ==========================================
# 4. FINE-TUNE THE CROSS-ENCODER RERANKER
# ==========================================
# Why: the reranker was zero-shot in the original pipeline. It never learned that,
# e.g., "5 Mind-Blowing Facts About X" should rerank highly against "A Formal Study
# of X: An Empirical Analysis". Training it on (query, pos)=1 / (query, hard-neg)=0
# pairs teaches exactly that mapping for your rewrite styles.
RERANKER_SAVE_PATH = "fine_tuned_reranker_v2"

if not os.path.exists(RERANKER_SAVE_PATH):
    print("\n[4/7] FINE-TUNING CROSS-ENCODER RERANKER...")
    reranker_examples = []
    for _, row in train_df.iterrows():
        query = row["generated_title"]  # no instruction prefix needed for cross-encoder
        pos = row["original_title"]
        negs = row.get("hard_negatives", []) or []
        reranker_examples.append(InputExample(texts=[query, pos], label=1.0))
        for neg in negs:
            reranker_examples.append(InputExample(texts=[query, neg], label=0.0))

    random.shuffle(reranker_examples)
    reranker_dataloader = DataLoader(reranker_examples, shuffle=True, batch_size=32)

    reranker = CrossEncoder("BAAI/bge-reranker-large", num_labels=1, device=DEVICE)
    reranker.fit(
        train_dataloader=reranker_dataloader,
        epochs=1,
        warmup_steps=int(0.1 * len(reranker_dataloader)),
        output_path=RERANKER_SAVE_PATH,
        show_progress_bar=True,
    )
    print(f"Reranker fine-tuning complete! Saved to '{RERANKER_SAVE_PATH}'")
else:
    print(f"\n[4/7] Found existing fine-tuned reranker at '{RERANKER_SAVE_PATH}'. Loading...")

reranker = CrossEncoder(RERANKER_SAVE_PATH, device=DEVICE)

# ==========================================
# 5. BUILD FAISS INDEX (with instruction-prefixed queries at search time)
# ==========================================
print("\n[5/7] Building FAISS Index with Fine-Tuned Embeddings...")
corpus_embeddings = dense_encoder.encode(
    corpus, batch_size=256, show_progress_bar=True, normalize_embeddings=True
)
dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(corpus_embeddings, dtype=np.float32))

print("Indexing BM25 Corpus...")
bm25 = BM25Okapi(bm25_corpus_tokens)

# ==========================================
# 6. HYBRID RETRIEVAL WITH RECIPROCAL RANK FUSION + WIDER CANDIDATE POOL
# ==========================================
# Why RRF instead of set union: raw cosine similarity (~0.3-0.9) and raw BM25 scores
# (unbounded, corpus-size dependent) live on incompatible scales. A straight union
# just means "whichever retriever happened to surface it" with no weighting of *how
# confidently* it was surfaced. RRF combines *rank position* from each retriever,
# which is scale-free and lets a strong dense rank compensate for a weak BM25 rank
# (exactly the situation for Catchy titles).
TOP_K_PER_RETRIEVER = 50  # widened from 30
RRF_K = 60  # standard RRF damping constant


def reciprocal_rank_fusion(rank_lists, k=RRF_K):
    fused_scores = {}
    for rank_list in rank_lists:
        for rank, idx in enumerate(rank_list, 1):
            fused_scores[idx] = fused_scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)


def run_pipeline(df, top_rerank=10):
    subtask1_rows, subtask2_rows = [], []
    total = len(df)
    print(f"\n[6/7] Executing Predictions on {total:,} Validation Samples...")

    for _, row in tqdm(df.iterrows(), total=total):
        q_id = row["id"]
        category = row.get("category", "unknown")
        gen_title = row["generated_title"]

        # Dense retrieval with instruction-prefixed query
        q_emb = dense_encoder.encode([QUERY_PREFIX + gen_title], normalize_embeddings=True)
        _, d_indices = index.search(np.array(q_emb, dtype=np.float32), TOP_K_PER_RETRIEVER)
        dense_rank_list = list(d_indices[0])

        # BM25 retrieval
        bm25_scores = bm25.get_scores(gen_title.lower().split())
        bm25_rank_list = list(np.argsort(bm25_scores)[::-1][:TOP_K_PER_RETRIEVER])

        # RRF fusion -> take top candidates for reranking
        fused = reciprocal_rank_fusion([dense_rank_list, bm25_rank_list])
        candidate_indices = [idx for idx, _ in fused[:TOP_K_PER_RETRIEVER]]
        candidate_titles = [corpus[idx] for idx in candidate_indices]

        # Fine-tuned cross-encoder reranking
        pairs = [[gen_title, cand] for cand in candidate_titles]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(candidate_titles, scores), key=lambda x: x[1], reverse=True)[:top_rerank]

        for rank_idx, (cand_title, score) in enumerate(ranked, 1):
            subtask1_rows.append(
                {"id": q_id, "rank": rank_idx, "score": round(float(score), 4), "original_title": cand_title}
            )

        subtask2_rows.append(
            {
                "id": q_id,
                "category": category,
                "generated_title": gen_title,
                "original_title": ranked[0][0],
            }
        )

    sub1_df = pd.DataFrame(subtask1_rows)
    sub2_df = pd.DataFrame(subtask2_rows)
    sub1_df.to_csv("subtask1_submission.tsv", sep="\t", index=False)
    sub2_df.to_csv("subtask2_submission.tsv", sep="\t", index=False)
    print("Saved 'subtask1_submission.tsv' and 'subtask2_submission.tsv'")
    return sub1_df, sub2_df


sub1_preds, sub2_preds = run_pipeline(val_df)

# ==========================================
# 7. EVALUATION (OVERALL + PER-CATEGORY)
# ==========================================
print("\n[7/7] Computing Evaluation Metrics...")


def calculate_mrr_at_k(preds_df, ground_truth_df, k=10, ids=None):
    gt_map = ground_truth_df.set_index("id")["original_title"].to_dict()
    grouped = preds_df.groupby("id")
    mrr_scores = []
    id_iter = ids if ids is not None else gt_map.keys()
    for q_id in id_iter:
        target_title = gt_map[q_id]
        if q_id not in grouped.groups:
            mrr_scores.append(0.0)
            continue
        group = grouped.get_group(q_id).sort_values("rank").head(k)
        predicted_titles = group["original_title"].tolist()
        rr = 0.0
        target_norm = str(target_title).strip().lower()
        for rank_idx, pred_title in enumerate(predicted_titles, 1):
            if str(pred_title).strip().lower() == target_norm:
                rr = 1.0 / rank_idx
                break
        mrr_scores.append(rr)
    return np.mean(mrr_scores) if mrr_scores else 0.0


def calculate_token_f1(pred_str, target_str):
    pred_tokens = re.findall(r"\w+", str(pred_str).lower())
    target_tokens = re.findall(r"\w+", str(target_str).lower())
    if not pred_tokens or not target_tokens:
        return 0.0
    common_tokens = set(pred_tokens) & set(target_tokens)
    num_same = sum(min(pred_tokens.count(w), target_tokens.count(w)) for w in common_tokens)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(target_tokens)
    return (2 * precision * recall) / (precision + recall)


overall_mrr = calculate_mrr_at_k(sub1_preds, val_df, k=10)

print("\n" + "=" * 55)
print("     ENHANCED PIPELINE EVALUATION SUMMARY            ")
print("=" * 55)
print(f" Overall MRR@10: {overall_mrr:.4f}")

if "category" in val_df.columns:
    print("\n Per-category MRR@10:")
    for cat, group in val_df.groupby("category"):
        cat_ids = group["id"].tolist()
        cat_mrr = calculate_mrr_at_k(sub1_preds, val_df, k=10, ids=cat_ids)
        flag = "  <-- target" if cat in HARD_CATEGORIES else ""
        print(f"   {cat:15s}: {cat_mrr:.4f}{flag}")

f1_scores = [calculate_token_f1(p, r) for p, r in zip(sub2_preds["original_title"], val_df["original_title"])]
mean_f1 = np.mean(f1_scores)

bertscore_metric = evaluate.load("bertscore")
# Pass device explicitly (single GPU) so bert_score doesn't attempt its own
# multi-GPU DataParallel wrapping internally.
bert_res = bertscore_metric.compute(
    predictions=sub2_preds["original_title"].tolist(),
    references=val_df["original_title"].tolist(),
    lang="en",
    device=DEVICE,
)
mean_bert = np.mean(bert_res["f1"])

print(f"\n Subtask 2 - Token F1 : {mean_f1:.4f}")
print(f" Subtask 2 - BERTScore: {mean_bert:.4f}")
print("=" * 55 + "\n")